# Group A synthesis: what are the decision models like?

Group A asks how the 17 decision models behave before any tabular predictor is involved: their outputs and typicality (A1), their sensitivity to presentation (A2) and how their output geometry relates to frozen encoders (A3, appendix). This notebook brings the three experiments together without redrawing their panels: one map places every model by values already shown in A1 and A2, and one table lists the full profile. It reads the same export ZIPs and recomputes nothing.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd

from src.evaluation import comparison_report, group_a, model_properties, study_export, study_reports
from src.utils.notebook_display import display_table
from src.utils.notebook_report import begin_report, finish_report, record, report_section
from src.visualize import figures, style
from src.visualize.group_b import render
from src.visualize.model_map import decision_model_map

style.apply()
NOTEBOOK = "A/A_synthesis"
save = figures.FigureSaver(NOTEBOOK)
begin_report()
report_section("1. What is complete")
result, a1_receipt = comparison_report.read_report(comparison_report.report_config(), allow_previous_roster=True)
a1 = a1_receipt["status"].startswith("verified")
a2_frames, a2_receipt = study_reports.robustness_export_tables()
a3_frames, a3_receipt = study_reports.geometry_export_tables()
print("A1:", comparison_report.describe_receipt(a1_receipt))
print("A2:", study_export.describe(a2_receipt))
print("A3:", study_export.describe(a3_receipt))
properties = model_properties.profile_table()
values = (group_a.a1_dataset_values(result.table("output_shape"), result.table("loo_consensus"), result.table("quality"))
          if a1 else pd.DataFrame(columns=["dataset_id", "task", "provider", "metric", "value"]))
excess = group_a.excess_summary(a2_frames["mode_effects"]) if a2_frames else pd.DataFrame(
    columns=["provider", "mode", "factor", "datasets", "mean"])
table = group_a.model_map_table(values, excess, properties)

A1: comparison.zip: created 2026-11-18T13:24:39.664228+00:00 by verified local export of 11 complete VSC models, 2026-11-18, 1,912,238 bytes, SHA-256 89c4986f83ba69546646a33b09ee4aa4ed8d94f9bd95f2d17050822d288cb8dd. Schema jevpfn.rq1a.report.v1 predates jevpfn.rq1a.report.v2; waiting for a new export. No measurements are shown.
A2: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\A\exp02_robustness\results\robustness\robustness.zip is absent. No measurements are shown.
A3: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\A\exp03_geometry\results\geometry\geometry.zip is absent. No measurements are shown.


## 2. One map of the decision models

(a) Typicality against stand-alone quality: how often a model gives the consensus top answer, against its median classification ROC-AUC (regression quality is rank-only and stays in the table). (b) Presentation sensitivity against confidence: the wording excess over replay on joint requests (A2) against the median top probability (A1). Marker shape is the architecture family, marker area the verified parameter count.

In [2]:
report_section("2. Decision-model map")
if not a1 and not a2_frames:
    print("Waiting for exports: the map needs the A1 comparison.zip and the A2 robustness.zip.")
else:
    if not a1:
        print("Waiting for the A1 export: panel (a) and the confidence axis of panel (b) are not drawn.")
    if not a2_frames:
        print("Waiting for the A2 export: panel (b) is not drawn.")
    render(save, decision_model_map(table))

Waiting for exports: the map needs the A1 comparison.zip and the A2 robustness.zip.


## 3. Profile table

One row per model: verified properties, the medians over datasets of the A1 joint-request measures (classification ROC-AUC and regression ρ over nine datasets each), and the A2 mean wording excess on joint requests. Missing measurements stay missing.

In [3]:
report_section("3. Profile table")
order = group_a.model_order(table.provider, properties)
shown = table.set_index("provider").reindex(order).reset_index()
shown.insert(0, "model", shown.provider.map(lambda p: style.DECISION_SHORT.get(p, p)))
display_table("Group A profile of the decision models (medians over datasets; missing = waiting for export or not stated)",
              shown.drop(columns="provider").round(3), preview_rows=17)
record(shown, "Group A profile table")
print("A3 geometry:", "waiting for export." if not a3_frames else
      f"{group_a.geometry_summary(a3_frames['comparisons']).provider.nunique()} models summarised in A/exp03_geometry.")

model,architecture_family,readout_type,parameters_billion,mean_top_probability,mean_normalized_entropy,consensus_agreement,classification_auc,regression_spearman,wording_excess_joint,wording_excess_datasets
Kev 27B,decoder_pointer_head,pointer_head,25.600,NaN,NaN,NaN,NaN,NaN,NaN,0
Clef-flash,decoder_pointer_head,pointer_head,9.410,NaN,NaN,NaN,NaN,NaN,NaN,0
Strands 2B,decoder_pointer_head,pointer_head,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
Liquid d1-3B,decoder_lm_head,lm_label_tokens,3.120,NaN,NaN,NaN,NaN,NaN,NaN,0
Intern 4B,decoder_lm_head,lm_label_tokens,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
SII 27B,decoder_lm_head,lm_label_tokens,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
Nimble 9B,decoder_lm_head,lm_label_tokens,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
Sieve 31B v3,bidirectional_encoder,None,31.000,NaN,NaN,NaN,NaN,NaN,NaN,0
Mammoth 27B,bidirectional_encoder,lm_label_tokens,27.000,NaN,NaN,NaN,NaN,NaN,NaN,0
Drex Max 27B,bidirectional_encoder,None,27.000,NaN,NaN,NaN,NaN,NaN,NaN,0


A3 geometry: waiting for export.


## Complete report

In [4]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the Group A export ZIPs and the model-properties configuration were read; nothing was computed on raw data.")

A/A_synthesis

1. What is complete


2. Decision-model map


3. Profile table


Group A profile of the decision models (medians over datasets; missing = waiting for export or not stated)
         model   architecture_family    readout_type  parameters_billion  mean_top_probability  mean_normalized_entropy  consensus_agreement  classification_auc  regression_spearman  wording_excess_joint  wording_excess_datasets
       Kev 27B  decoder_pointer_head    pointer_head              25.600                   NaN                      NaN                  NaN                 NaN                  NaN                   NaN                        0
    Clef-flash  decoder_pointer_head    pointer_head               9.410                   NaN                      NaN                  NaN                 NaN                  NaN                   NaN                        0
    Strands 2B  decoder_pointer_head    pointer_head                 NaN                   NaN                      NaN       